In [1]:
import time
import pandas as pd
from scipy.sparse import load_npz, hstack
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
import nltk
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
import optuna

Se carga dataset.

In [2]:
X_train = load_npz('datasets/X_train.npz')
X_test = load_npz('datasets/X_test.npz')
y_train = pd.read_csv('datasets/y_train.csv')['label']
y_test = pd.read_csv('datasets/y_test.csv')['label']

Se agregan los tres features más significativos recomendados en la bibliografía útiles para este tipo de algoritmos; se sugieren muchos más, pero también se sugiere que se use la menor cantidad de features extra para alcanzar la mejor eficiencia del modelo.

In [3]:
X_extra_train = pd.read_csv('datasets/X_extra_train.csv')
X_extra_test = pd.read_csv('datasets/X_extra_test.csv')

### Cantidad de verbos imperativos
Cuenta la presencia de verbos directivos o de mando en el correo electrónico. En la bibliografía, los verbos específicos considerados son "click", "verify",
"submit", "download" y "update").

In [4]:
IMPERATIVE_VERBS = ['click', 'verify', 'submit', 'download', 'update']

def get_imperative_verb_count(text):
    return sum(text.lower().count(verb) for verb in IMPERATIVE_VERBS)

X_extra_train['imperative_verbs_count'] = X_extra_train['remainder__Email Text Raw'].apply(get_imperative_verb_count)
X_extra_test['imperative_verbs_count'] = X_extra_test['remainder__Email Text Raw'].apply(get_imperative_verb_count)

### Complejidad en oraciones
Considera el número de conectores por oración. Se divide por el número total de oraciones, lo que permite medir cuántas oraciones subordinadas pueden aparecer en todo el texto.

In [5]:
LINKING_WORDS = ['and', 'but', 'or', 'because']

nltk.data.path.append('nltk_data')
nltk.download('punkt_tab', 'nltk_data')

def get_clause_density(text):
    sentence_count = len(sent_tokenize(text))
    if sentence_count == 0:
        return 0.0
    total_conjunctions = sum(text.lower().count(word) for word in LINKING_WORDS)
    complexity_ratio = total_conjunctions / sentence_count
    return complexity_ratio / sentence_count

X_extra_train['clause_density'] = X_extra_train['remainder__Email Text Raw'].apply(get_clause_density)
X_extra_test['clause_density'] = X_extra_test['remainder__Email Text Raw'].apply(get_clause_density)

[nltk_data] Downloading package punkt_tab to nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


### Densidad de pronombres
Divide el número total de pronombres entre el número total de palabras.

In [6]:
TARGET_PRONOUNS = ['i', 'you', 'he', 'she', 'it', 'we', 'they']

def get_pronoun_density(text):
    tokens = word_tokenize(text.lower())
    total_words = len(tokens)
    if total_words == 0:
        return 0.0
    pronoun_count = sum(1 for word in tokens if word in TARGET_PRONOUNS)
    return pronoun_count / total_words

X_extra_train['pronoun_density'] = X_extra_train['remainder__Email Text Raw'].apply(get_pronoun_density)
X_extra_test['pronoun_density'] = X_extra_test['remainder__Email Text Raw'].apply(get_pronoun_density)

KNN requiere valores estandarizados.

In [7]:
extra_cols = ['imperative_verbs_count', 'clause_density', 'pronoun_density']

X_train = hstack([X_train, X_extra_train[extra_cols].values])
X_test = hstack([X_test, X_extra_test[extra_cols].values])

scaler = StandardScaler(with_mean=False)

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Se inicializa KNN con _k = 5_ y se prueba.

In [8]:
knn_model = KNeighborsClassifier(n_neighbors=5)

start_train_time = time.time()
knn_model.fit(X_train_scaled, y_train)
end_train_time = time.time()

train_duration = end_train_time - start_train_time

start_predict_duration = time.time()
y_pred = knn_model.predict(X_test_scaled)
end_predict_duration = time.time()

predict_duration = end_predict_duration - start_predict_duration

Se evalúa.

In [9]:
print(f"Training duration: {train_duration:.4f}")
print(f"Predict duration: {predict_duration:.4f}")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

Training duration: 0.0070
Predict duration: 2.7090
Accuracy: 0.5113
              precision    recall  f1-score   support

           0       0.98      0.22      0.36      1979
           1       0.44      0.99      0.61      1207

    accuracy                           0.51      3186
   macro avg       0.71      0.61      0.48      3186
weighted avg       0.77      0.51      0.45      3186



## Optimización de parámetros

Se define la función objetivo y se prepara el estudio.

In [ ]:
def objective(trial):
    n_neighbors = trial.suggest_int("n_neighbors", 1, 50)
    weights = trial.suggest_categorical("weights", ["uniform", "distance"])
    metric = trial.suggest_categorical("metric", ["euclidean", "manhattan", "cosine"])

    model = KNeighborsClassifier(n_neighbors=n_neighbors, weights=weights, metric=metric)

    score = cross_val_score(model, X_train_scaled, y_train, cv=3, scoring='f1', n_jobs=-1)
    return score.mean()

def champion_callback(study, frozen_trial):
    winner = study.user_attrs.get("winner", None)

    if study.best_value and winner != study.best_value:
        study.set_user_attr("winner", study.best_value)
        if winner:
            improvement_percent = (abs(winner - study.best_value) / study.best_value)
            improvement_percent *= 100
            print(
                f"Trial {frozen_trial.number} achieved value: {frozen_trial.value} "
                f"with {improvement_percent: .4f}% improvement"
            )
        else:
            print(f"Initial trial {frozen_trial.number} achieved value: " 
            f"{frozen_trial.value}")

optuna.logging.set_verbosity(optuna.logging.ERROR)

Se ejecutan 500 trials.

In [21]:
study = optuna.create_study(direction="maximize", pruner=optuna.pruners.HyperbandPruner)
study.optimize(objective, n_trials=100, callbacks=[champion_callback], n_jobs=-1)

Initial trial 2 achieved value: 0.9103518282332792
Trial 5 achieved value: 0.916369340596915 with  0.6567% improvement
Trial 29 achieved value: 0.917893735782994 with  0.1661% improvement


In [22]:
best_params = study.best_params
best_params

{'n_neighbors': 50, 'weights': 'distance', 'metric': 'cosine'}

Se entrena el modelo con los parámetros resultantes de la optimización.

In [23]:
optimized_knn_model = KNeighborsClassifier(**best_params)

start_train_time = time.time()
optimized_knn_model.fit(X_train_scaled, y_train)
end_train_time = time.time()

train_duration = end_train_time - start_train_time

start_predict_duration = time.time()
y_pred = optimized_knn_model.predict(X_test_scaled)
end_predict_duration = time.time()

predict_duration = end_predict_duration - start_predict_duration

Se evalúa.

In [24]:
print(f"Training duration: {train_duration:.4f}")
print(f"Predict duration: {predict_duration:.4f}")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

Training duration: 0.0112
Predict duration: 1.7689
Accuracy: 0.9319
              precision    recall  f1-score   support

           0       0.93      0.97      0.95      1979
           1       0.94      0.87      0.91      1207

    accuracy                           0.93      3186
   macro avg       0.93      0.92      0.93      3186
weighted avg       0.93      0.93      0.93      3186

